# ED Alignment Algorithm on Real Piano Performances

This notebook evaluates the ED alignment algorithm in `compareMusic` on real piano performances from the [(n)ASAP: the (note-)Aligned Scores And Performances dataset](https://github.com/CPJKU/asap-dataset) (Peter et al., 2023).

(n)ASAP is a dataset of aligned musical scores and performances built by extending the ASAP dataset with note-level annotations. The ASAP contains 236 distinct musical scores and 1067 performances of Western classical piano music from 15 different composers, the piece directory contains the XML and MIDI score, plus all of the performances of a specific piece, including: 
- `midi_score`: the MIDI score (what should be played) — used as **reference**
- `midi_performance`: what the pianist actually played — used as **response**
- `note_alignments.tsv`: official note-level alignment annotations — used as **ground truth**

bib citation:

@article{Peter-2023,
 title = {Automatic Note-Level Score-to-Performance Alignments in the ASAP Dataset},
 author = {Peter, Silvan David and Cancino-Chacón, Carlos Eduardo and Foscarin, Francesco and McLeod, Andrew Philip and Henkel, Florian and Karystinaios, Emmanouil and Widmer, Gerhard},
 doi = {10.5334/tismir.149},
 journal = {Transactions of the International Society for Music Information Retrieval {(TISMIR)}},
 year = {2023}
}

relevant paper: https://transactions.ismir.net/articles/10.5334/tismir.149#5-alignment-of-the-asap-dataset

## Download the (n)ASAP Dataset

In [17]:
import os

# Note: use CPJKU version (not fosfrancesco) because only CPJKU has
# the note_alignments TSV files are needed for ground truth comparison.
if not os.path.exists("asap-dataset"):
    os.system("git clone https://github.com/CPJKU/asap-dataset.git")
else:
    print("asap-dataset already exists, skipping download.")

ASAP_PATH = "asap-dataset"

asap-dataset already exists, skipping download.


## Load the Ground Truth (GT)

The `note_alignment.tsv` file in each performance contains the official note-level alignment annotations. Each row represents one note, with columns: `xml_id`, `midi_id`, `track`, `channel`, `pitch`, `onset`.
- If `midi_id == "deletion"`: the score note was not played → label `deletion`
- If `xml_id == "insertion"`: an extra note was played → label `insertion`
- Otherwise: a matched pair → label `paired`

For `match` and `insertion` rows, the TSV provides `onset` (onset time in seconds) and `pitch` (MIDI pitch) for the performance note.

In [18]:
import csv

def load_ground_truth(tsv_path):
    """
    Read a note_alignments TSV file from the CPJKU ASAP dataset.

    Args:
        tsv_path: str, path to the note_alignments.tsv file

    Returns:
        list of dicts, each with keys:
            label  -> 'paired', 'insertion', or 'deletion'
            onset  -> float (seconds) or None for deletions
            pitch  -> int (MIDI pitch number) or None for deletions
    """
    rows = []
    with open(tsv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        
        for row in reader:
            xml_id  = row["xml_id"].strip()
            midi_id = row["midi_id"].strip()

            if midi_id == "deletion":
                rows.append({"label": "deletion", "onset": None, "pitch": None})
            elif xml_id == "insertion":
                rows.append({
                    "label": "insertion",
                    "onset": float(row["onset"]),
                    "pitch": int(row["pitch"]),
                })
            else:
                rows.append({
                    "label": "paired",
                    "onset": float(row["onset"]),
                    "pitch": int(row["pitch"]),
                })
    return rows

## Load Score (reference)/Performance (response) pairs

Helper functions for format conversion: These functions convert a MIDI file into the `{pitch, start, duration}` format used by `compare_MIDI.py`.

In [19]:
import pretty_midi

def midi_file_to_notes(midi_path):
    """
    Parse a MIDI file and return all its notes in compareMusic format.
    
    Args:
        midi_path: str, path to the MIDI file

    Returns:
        list of dicts sorted by onset time, each with keys:
            pitch -> int, MIDI note number
            start -> float, onset time in seconds
            duration -> float, note length in seconds
    """
    midi_data = pretty_midi.PrettyMIDI(midi_path)
    all_notes = []
    for instrument in midi_data.instruments:
        if instrument.is_drum: # Ignore drum tracks
            continue
        for note in instrument.notes:
            all_notes.append({
                "pitch": note.pitch,
                "start": round(note.start, 3),
                "duration": round(note.end - note.start, 3),
            })
    # Sort by onset time, then by pitch (ensures consistent ordering for chords)
    all_notes.sort(key=lambda n: (n["start"], n["pitch"]))
    return all_notes


def build_sample(ref_path, response_path, composer, title, metadata_row):
    """
    Convert one score/performance MIDI pair into a sample dict
    ready for compare_performance_ED.

    Args:
        ref_path: str, path to the score (reference) MIDI file
        response_path: str, path to the performance (response) MIDI file
        composer: str
        title: str
        metadata_row: dict, one row from metadata.csv

    Returns:
        sample dict with keys: composer, title, reference, response, metadata_row
        Returns None if either MIDI file produces zero notes.
    """
    score_notes = midi_file_to_notes(ref_path)
    perf_notes  = midi_file_to_notes(response_path)

    # If either MIDI file has no notes, return None to indicate that this sample
    # should be skipped (e.g., some ASAP pieces have empty score or performance).
    if not score_notes or not perf_notes:
        return None

    return {
        "composer": composer,
        "title": title,
        "reference": {"notes": score_notes},
        "response": {"notes": perf_notes},
        "metadata_row": metadata_row,
    }

For the (n)ASAP dataset, the `metadata.csv` lists every score/performance pair in the dataset, check that both MIDI files exist on disk.

In [ ]:
def load_samples(asap_path, composer=None, max_notes=8000):
    """
    Read the ASAP metadata CSV and return a list of sample dicts
    for a specific composer only.

    Args:
        asap_path:  str, path to the cloned ASAP repo root
        composer:   str or None.
                    If a string (e.g. "Bach"), only that composer is loaded.
                    If None, all composers in the dataset are loaded.
        max_notes:  int, skip any pair where either the score or performance
                    has more than this many notes. Default 3000.
                    Set to None to disable the limit.

    Returns:
        list of sample dicts (see build_sample)
    """
    metadata_path = os.path.join(asap_path, "metadata.csv")
    samples = []
    skipped = 0

    with open(metadata_path, "r", encoding="utf-8") as csv_file:
        reader = csv.DictReader(csv_file)

        for row in reader:
            row_composer = row.get("composer", "").strip()
            # Skip rows that do not match the requested composer (if one is given)
            if composer is not None and row_composer != composer:
                continue

            ref_path = os.path.join(asap_path, row.get("midi_score", "").strip())
            response_path = os.path.join(asap_path, row.get("midi_performance", "").strip())

            if os.path.isfile(ref_path) and os.path.isfile(response_path):
                sample = build_sample(
                    ref_path, response_path,
                    row_composer,     
                    row.get("title", "Unknown"),
                    dict(row),
                    )

                if sample is not None:
                    ref_count  = len(sample["reference"]["notes"])
                    perf_count = len(sample["response"]["notes"])
                    exceeds_limit = (
                        max_notes is not None
                        and (ref_count > max_notes or perf_count > max_notes)
                    )
                    if exceeds_limit:                            
                        skipped = skipped + 1
                    else:
                        samples.append(sample)

    print(f"Skipped {skipped} pairs due to exceeding max_notes limit of {max_notes}")
    print(f"Loaded {len(samples)} score (reference) / performance (response) pairs.")
    return samples

samples = load_samples(ASAP_PATH, composer=None, max_notes=8000)

Skipped 45 pairs due to exceeding max_notes limit of 8000
Loaded 1021 score (reference) /performance (response) pairs.


# Run Alignment on Each Pair

Passing each score/performance pair through `compare_performance_ED`, where score MIDI is the reference and the performance MIDI is the response.

**Why do we need the normalised events? - to fix the onset mismatch bug**
- Inside `compare_performance_ED`, `normalize_start_times` shifts the first note to t = 0, and `group_notes_into_events` groups simultaneous notes into chords. The `response_index` values stored in `event_details` refer to positions in these grouped events, not in the original flat note list. Therefore, need to reproduce these two steps here so that the correct onset and pitch for each event during ground truth comparison can be looked up.
- The `response_onset_offset` (the original first-note onset before normalisation) will also be recorded, and will be added back when comparing normalised pipeline onsets against the original (absolute) times stored in the ground truth TSV.

In [22]:
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from asap_worker import evaluate_one_sample


def run_alignment_on_samples_parallel(samples, max_workers=None):
    """
    Run evaluate_one_sample on every sample using a process pool, so multiple
    pieces are aligned at the same time on different CPU cores.

    Args:
        samples: list of sample dicts from load_samples()
        max_workers: int or None. None means "all CPU cores minus one",
                     leaving one core free for the OS and other apps.

    Returns:
        results: list of successful result dicts
        failed: list of (composer, title, error_message) for pieces that failed
    """
    if max_workers is None:
        max_workers = max(1, os.cpu_count() - 1)

    results = []
    failed = []

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_to_sample = {
            executor.submit(evaluate_one_sample, sample): sample
            for sample in samples
        }

        for future in tqdm(as_completed(future_to_sample), total=len(samples)):
            sample = future_to_sample[future]
            result = future.result()  # exceptions are already caught inside the worker

            if result["error"] is not None:
                failed.append((result["composer"], result["title"], result["error"]))
            else:
                results.append(result)

    print("Succeeded:", len(results), " Failed:", len(failed))
    return results, failed


all_results, failed_pieces = run_alignment_on_samples_parallel(samples)

100%|██████████| 1021/1021 [15:15<00:00,  1.11it/s] 


Succeeded: 1021  Failed: 0


## Compute Precision, Recall, and F1 

**our pipeline vs GT**

The GT has three different labels, while our pipeline has four operations:

| Pipeline operation | GT label | Explanation |
|---|---|---|
| `match` | `paired` | Same pitch — GT still calls this `paired` |
| `replacement` | `paired` | Different pitch — GT still calls this `paired`, because the score note *was* played at approximately the right time |
| `missing` | `deletion` | Score note not found in response |
| `extra` | `insertion` | Response note has no score counterpart |

**Evaluation metrics design**

GT does not distinguish `match` from `replacement` because the alignment methods in GT are based on the temporal information. However, the cost function in our edit distance algorithm is based on the pitch correctness. 

Following Peter et al. (2023, TISMIR), label-level F1 would be applied as the main metric where TP, FP, and FN are computed separately for each label type and then summed:

| Label | Matching key | TP | FP | FN |
|---|---|---|---|---|
| `paired` (pipeline `match` / `replacement`) | onset only (rounded to 1 d.p.) | Pipeline and ground truth agree a score note was aligned to a performance note at this onset (regardless of pitch correctness) | Pipeline predicts a pairing at this onset that the ground truth does not recognise | Ground truth expects a pairing at this onset that the pipeline fails to find |
| `deletion` (pipeline `missing`) | count only as no onset or pitch available for deletions | Overlap between the number of notes the pipeline judges as missing and the number the ground truth judges as missing | Pipeline predicts more missing notes than the ground truth; the excess | Ground truth expects more missing notes than the pipeline predicts; the shortfall |
| `insertion` (pipeline `extra`) | `(pitch, onset)` | Pipeline and ground truth agree an extra, unexpected note was played at this pitch and onset | Pipeline flags a note as extra that the ground truth does not agree with | Ground truth marks a note as extra that the pipeline fails to identify |

- Precision: TP / (TP + FP), i.e. of all labels the pipeline predicted, how many were correct  
- Recall: TP / (TP + FN), i.e. of all GT labels, how many the pipeline found  
- F1: the harmonic mean of precision and recall 

In [23]:
def convert_pipeline_output(event_details, response_events_normalized):
    """
    Convert event_details from compare_performance_ED into a list of
    dicts with keys: onset, pitch, label.
    Label mapping from pipeline operations to GT labels:
        match or replacement -> 'paired'
        missing -> 'deletion'
        extra -> 'insertion'

    Chord events are expanded note-by-note using correct_pitches /
    missing_pitches / extra_pitches from event_level_feedback.

    Args:
        event_details: list of dicts from compare_performance_ED
        response_events_normalized: list of event dicts produced by
                                    group_notes_into_events on the normalised
                                    response notes — NOT the flat note list.

    Returns:
        list of dicts with keys:
            onset -> float (normalised seconds) or None for deletions
            pitch -> int (MIDI pitch) or None for deletions
            label -> 'paired', 'insertion', or 'deletion'
    """
    my_pairs = []

    for event in event_details:
        op = event["operation_type"]

        if event["event_type"] == "note":
            if op in ("match", "replacement"):
                # Look up onset and pitch from the grouped event list.
                ev = response_events_normalized[event["response_index"] - 1]
                my_pairs.append({
                    "onset": ev["event_start"],
                    "pitch": ev["notes"][0]["pitch"],
                    "label": "paired",
                })
            elif op == "missing":
                my_pairs.append({"onset": None, "pitch": None, "label": "deletion"})
            elif op == "extra":
                ev = response_events_normalized[event["response_index"] - 1]
                my_pairs.append({
                    "onset": ev["event_start"],
                    "pitch": ev["notes"][0]["pitch"],
                    "label": "insertion",
                })

        elif event["event_type"] == "chord":
            if op == "missing":
                # Entire chord missing: one deletion per note in the ref chord
                # correct_pitches + missing_pitches = all ref notes in this chord
                num_ref_notes = (
                    len(event["correct_pitches"] or [])
                    + len(event["missing_pitches"] or [])
                )
                for i in range(num_ref_notes):
                    my_pairs.append({"onset": None, "pitch": None, "label": "deletion"})
            elif op == "extra":
                # Every note in the extra response chord counts as an insertion.
                ev = response_events_normalized[event["response_index"] - 1]
                for note in ev["notes"]:
                    my_pairs.append({
                        "onset": ev["event_start"],
                        "pitch": note["pitch"],
                        "label": "insertion",
                    })
            else:
                # Aligned chord pair (match or replacement).
                ev = response_events_normalized[event["response_index"] - 1]
                onset = ev["event_start"]

                # Build a pitch-class -> MIDI-pitch lookup from the response chord.
                # This fixes the bug where the original code always used the first
                # note's pitch regardless of which pitch class was being expanded.
                pc_to_midi = {}
                for note in ev["notes"]:
                    pc = note["pitch"] % 12
                    pc_to_midi[pc] = note["pitch"]

                # Correctly matched pitch classes -> paired
                for pc in (event["correct_pitches"] or []):
                    midi_pitch = pc_to_midi.get(pc, pc)  # fallback if lookup fails
                    my_pairs.append({"onset": onset, "pitch": midi_pitch, "label": "paired"})

                # Reference pitch classes not played -> deletion
                for pc in (event["missing_pitches"] or []):
                    my_pairs.append({"onset": None, "pitch": None, "label": "deletion"})

                # Response pitch classes with no reference counterpart -> insertion
                for pc in (event["extra_pitches"] or []):
                    midi_pitch = pc_to_midi.get(pc, pc)
                    my_pairs.append({"onset": onset, "pitch": midi_pitch, "label": "insertion"})

    return my_pairs

In [24]:
def count_paired_matches(ground_truth, my_pairs, onset_offset):
    """
    Count TP, FP, FN for 'paired' labels.

    Matching key: onset only (rounded to 1 d.p.).
    Pitch is NOT used here because the GT 'paired' label means the score note
    was aligned to a performance note regardless of whether the pitch is correct.
    Both pipeline 'match' and 'replacement' operations map to 'paired'.

    Args:
        ground_truth: list of dicts from load_ground_truth()
        my_pairs: list of dicts from convert_pipeline_output()
        onset_offset: float, added back to normalised onsets before comparison

    Returns:
        tp, fp, fn: ints
    """
    # Collect GT 'paired' onset keys
    gt_paired_set = set()
    for row in ground_truth:
        if row["label"] == "paired":
            gt_paired_set.add(round(float(row["onset"]), 1))

    # Collect predicted 'paired' onset keys, restoring absolute time
    my_paired_set = set()
    for row in my_pairs:
        if row["label"] == "paired" and row["onset"] is not None:
            absolute_onset = row["onset"] + onset_offset
            my_paired_set.add(round(float(absolute_onset), 1))

    tp = len(my_paired_set & gt_paired_set)  # correctly predicted
    fp = len(my_paired_set - gt_paired_set)  # predicted but not in GT
    fn = len(gt_paired_set - my_paired_set)  # in GT but not predicted

    return tp, fp, fn


def count_deletion_matches(ground_truth, my_pairs):
    """
    Count TP, FP, FN for 'deletion' labels.

    Deletions have no onset or pitch (the note was never played), so
    identity-based matching is not possible. Count-based matching is used:
        TP = min(GT deletion count, predicted deletion count)
        FP = max(0, predicted count - GT count)
        FN = max(0, GT count - predicted count)

    Args:
        ground_truth: list of dicts from load_ground_truth()
        my_pairs: list of dicts from convert_pipeline_output()

    Returns:
        tp, fp, fn: ints
    """
    gt_deletion_count = sum(1 for row in ground_truth if row["label"] == "deletion")
    my_deletion_count = sum(1 for row in my_pairs if row["label"] == "deletion")

    tp = min(gt_deletion_count, my_deletion_count)
    fp = max(0, my_deletion_count - gt_deletion_count)
    fn = max(0, gt_deletion_count - my_deletion_count)

    return tp, fp, fn


def count_insertion_matches(ground_truth, my_pairs, onset_offset):
    """
    Count TP, FP, FN for 'insertion' labels.

    Matching key: (pitch, onset) -- pitch IS included here because GT
    insertion records carry a pitch value, and we want to verify that
    the correct extra note was identified.

    Args:
        ground_truth: list of dicts from load_ground_truth()
        my_pairs: list of dicts from convert_pipeline_output()
        onset_offset: float, added back to normalised onsets before comparison

    Returns:
        tp, fp, fn: ints
    """
    # Collect GT 'insertion' (pitch, onset) keys
    gt_insertion_set = set()
    for row in ground_truth:
        if row["label"] == "insertion":
            key = (int(row["pitch"]), round(float(row["onset"]), 1))
            gt_insertion_set.add(key)

    # Collect predicted 'insertion' (pitch, onset) keys, restoring absolute time
    my_insertion_set = set()
    for row in my_pairs:
        if row["label"] == "insertion" and row["onset"] is not None:
            absolute_onset = row["onset"] + onset_offset
            key = (int(row["pitch"]), round(float(absolute_onset), 1))
            my_insertion_set.add(key)

    tp = len(my_insertion_set & gt_insertion_set)
    fp = len(my_insertion_set - gt_insertion_set)
    fn = len(gt_insertion_set - my_insertion_set)

    return tp, fp, fn

In [25]:
def compute_metrics_label_level(ground_truth, my_pairs, onset_offset=0.0):
    """
    Compute label-level precision, recall, and F1 following Peter et al. (2023).
    Args:
        ground_truth: list of dicts from load_ground_truth()
        my_pairs: list of dicts from convert_pipeline_output()
        onset_offset: float, the first-note onset before normalisation.
                      Added back to convert normalised onsets to absolute time.

    Returns:
        dict with keys:
            precision, recall, f1 -> floats (rounded to 4 d.p.)
            tp, fp, fn  -> ints (totals across all labels)
            tp_paired, fp_paired, fn_paired
            tp_deletion, fp_deletion, fn_deletion
            tp_insertion, fp_insertion, fn_insertion
    """
    # Count TP/FP/FN separately for each label type
    tp_p, fp_p, fn_p = count_paired_matches(ground_truth, my_pairs, onset_offset)
    tp_d, fp_d, fn_d = count_deletion_matches(ground_truth, my_pairs)
    tp_i, fp_i, fn_i = count_insertion_matches(ground_truth, my_pairs, onset_offset)

    # Sum across all label types for overall metrics
    tp = tp_p + tp_d + tp_i
    fp = fp_p + fp_d + fp_i
    fn = fn_p + fn_d + fn_i

    # Precision: of all labels predicted, how many are correct
    if tp + fp > 0:
        precision = tp / (tp + fp)
    else:
        precision = 0.0

    # Recall: of all GT labels, how many the pipeline found
    if tp + fn > 0:
        recall = tp / (tp + fn)
    else:
        recall = 0.0

    # F1: harmonic mean of precision and recall
    if precision + recall > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0.0

    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp, "fp": fp, "fn": fn,
        # Per-label breakdown -- useful for diagnosing which label type is weakest
        "tp_paired": tp_p, "fp_paired": fp_p, "fn_paired": fn_p,
        "tp_deletion": tp_d, "fp_deletion": fp_d, "fn_deletion": fn_d,
        "tp_insertion": tp_i, "fp_insertion": fp_i, "fn_insertion": fn_i,
    }

Evaluate against ground truth:

In [32]:
def evaluate_one_piece(asap_path, metadata_row, event_details,
                       response_events_normalized, response_onset_offset):
    """
    Run ground truth comparison for each score/performance pair.

    Args:
        asap_path: str, path to ASAP repo root
        metadata_row: dict, one row from metadata.csv
        event_details: list from compare_performance_ED result
        response_events_normalized: list of event dicts (normalized + grouped)
        response_onset_offset: float, onset of the first response note
                               before normalization — added back to convert
                               normalised onsets to absolute times for GT comparison.

    Returns:
        metrics dict, or None if the TSV file is not found
    """
    tsv_rel = metadata_row.get("note_alignments", "").strip()
    tsv_path = os.path.join(asap_path, tsv_rel)

    if not os.path.isfile(tsv_path):
        # No print here -- the caller counts and reports how many were
        # skipped in a single summary line instead.
        return None

    ground_truth = load_ground_truth(tsv_path)
    my_pairs = convert_pipeline_output(event_details, response_events_normalized)
    metrics = compute_metrics_label_level(ground_truth, my_pairs, onset_offset=response_onset_offset)

    return metrics

In [35]:
import pandas as pd

eval_rows = []
skipped_count = 0

for result in all_results:
    metrics = evaluate_one_piece(
        ASAP_PATH,
        result["metadata_row"],
        result["event_details"],
        result["response_events_normalized"],
        result["response_onset_offset"],
    )
    if metrics is not None:
        eval_rows.append({
            "Piece": result["composer"] + " (" + result["title"] + ")",
            "Precision": metrics["precision"],
            "Recall": metrics["recall"],
            "F1": metrics["f1"],
            "TP": metrics["tp"],
            "FP": metrics["fp"],
            "FN": metrics["fn"],
        })
    else:
        skipped_count += 1

print(f"Skipped {skipped_count} piece(s) due to missing TSV file.")

df_eval = pd.DataFrame(eval_rows)
display(df_eval.head(10))

print("Mean Precision:", round(df_eval["Precision"].mean(), 4))
print("Mean Recall :", round(df_eval["Recall"].mean(), 4))
print("Mean F1 :", round(df_eval["F1"].mean(), 4))

Skipped 30 piece(s) due to missing TSV file.


,Piece,Precision,Recall,F1,TP,FP,FN
0,Bach (Fugue_bwv_846),0.9463,0.9136,0.9297,423,24,40
1,Bach (Fugue_bwv_848),0.9675,0.9260,0.9463,864,29,69
2,Bach (Fugue_bwv_848),0.9850,0.9500,0.9672,855,13,45
3,Bach (Fugue_bwv_848),0.9840,0.9378,0.9604,860,14,57
4,Bach (Fugue_bwv_848),0.9522,0.9232,0.9375,877,44,73
5,Bach (Fugue_bwv_848),0.9621,0.9162,0.9386,864,34,79
6,Bach (Fugue_bwv_848),0.9873,0.9326,0.9592,858,11,62
7,Bach (Fugue_bwv_848),0.9861,0.9193,0.9515,854,12,75
8,Bach (Fugue_bwv_848),0.9809,0.9083,0.9432,872,17,88
9,Bach (Fugue_bwv_848),0.9874,0.9330,0.9594,863,11,62


Mean Precision: 0.8898
Mean Recall : 0.8727
Mean F1 : 0.8793
